## AGENDA
1. how to integrate our custom tokens into the tokenizer of the base model
2. How to create a chat template using the custom tokens that we have created and seemlessly integrate our data for efficient processing

In [1]:
!pip install -q -U transformers datasets bitsandbytes trl peft huggingface_hub

In [2]:
from huggingface_hub import notebook_login

In [3]:
notebook_login()

In [7]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import torch

In [11]:
model_name = "mistralai/Mistral-7B-v0.3"

config_4bit = BitsAndBytesConfig(load_in_4bit=True)

model_4bit = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = config_4bit,
    device_map= "auto",
    trust_remote_code= True
)

Tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left", trust_remote_code=True)

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/137k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [12]:
from datasets import load_dataset

dataset = load_dataset("MattCoddity/dockerNLcommands")
dataset

README.md:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

06102023.json:   0%|          | 0.00/543k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2415 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 2415
    })
})

In [17]:
import copy
tokenizer = copy.deepcopy(Tokenizer)
tokenizer.all_special_tokens

['<s>', '</s>', '<unk>']

In [18]:
tokens = ["<system>","<user>", "<assistant>" ]


for token in tokens:
    token_id = tokenizer.encode(token, add_special_tokens=False)
    print(token, token_id)

<system> [1291, 7342, 29535]
<user> [1291, 2606, 29535]
<assistant> [1291, 1257, 11911, 29535]


In [19]:
tokenizer.pad_token

In [20]:
special_token = {
    "pad_token": "<PAD>",
    "additional_special_tokens": ["<system>", "<user>", "<assistant>"]
}

tokenizer.add_special_tokens(special_token)

4

In [21]:
tokens = ["<system>","<user>", "<assistant>" ]


for token in tokens:
    token_id = tokenizer.encode(token, add_special_tokens=False)
    print(token, token_id)

<system> [32769]
<user> [32770]
<assistant> [32771]


In [22]:
len(tokenizer)

32772

In [24]:
model_4bit.config.vocab_size

32768

In [25]:
model_4bit.resize_token_embeddings(len(tokenizer))

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
[transformers] The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(32772, 4096)

In [26]:
tokenizer.all_special_tokens

['<s>', '</s>', '<unk>', '<PAD>', '<system>', '<user>', '<assistant>']

#### Creating Template with New custom Tokens and integrating them into the dataset

In [27]:
tokenizer.bos_token

'<s>'

In [28]:
tokenizer.eos_token

'</s>'

In [32]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 2415
    })
})

In [33]:
new_template = """<s><system>{system_prompt}</s><user>{user_prompt}</s><assistant>{model_answer}</s>"""

def format_dataset(example):
    system_prompt = example["instruction"]
    user_prompt = example["input"]
    model_answer = example["output"]

    formatted_text = new_template.format(
        system_prompt = system_prompt,
        user_prompt = user_prompt,
        model_answer = model_answer
    )

    return {"text":formatted_text}

    

In [35]:
sample = dataset["train"][0]
sample

{'input': 'Give me a list of containers that have the Ubuntu image as their ancestor.',
 'output': "docker ps --filter 'ancestor=ubuntu'",
 'instruction': 'translate this sentence in docker command'}

In [36]:
formatted_text = format_dataset(sample)
formatted_text

{'text': "<s><system>translate this sentence in docker command</s><user>Give me a list of containers that have the Ubuntu image as their ancestor.</s><assistant>docker ps --filter 'ancestor=ubuntu'</s>"}

In [37]:
dataset = dataset.map(format_dataset)


Map:   0%|          | 0/2415 [00:00<?, ? examples/s]

In [41]:
dataset["train"][10]

{'input': 'I want to see a list of all Docker images, including the dangling ones. Could you please provide their repository, tags, and IDs as well?',
 'output': 'docker images -a --format "{{.Repository}},{{.Tag}},{{.ID}}"',
 'instruction': 'translate this sentence in docker command',
 'text': '<s><system>translate this sentence in docker command</s><user>I want to see a list of all Docker images, including the dangling ones. Could you please provide their repository, tags, and IDs as well?</s><assistant>docker images -a --format "{{.Repository}},{{.Tag}},{{.ID}}"</s>'}